# fedOPD data fetch

Three datasets, each in its own folder under `../data`:

| # | dataset | folder |
|---|---------|--------|
| 1 | NHS RTT monthly Full CSV zips, Apr 2018 – Mar 2026 (96 files, ~8 GB extracted) | `data/rtt_zips` -> `data/rtt_csv` |
| 2 | Trust -> ICB / geography reference (ODS ORD API) | `data/reference` |
| 3 | KH03 overnight bed availability & occupancy | `data/kh03` |



In [1]:
import re                                   # regex engine: pulls the month token out of messy NHS filenames
import zipfile                              # reads/extracts .zip archives without shelling out to unzip
from pathlib import Path                    # path objects: join with /, .exists(), works on Windows and Linux
from datetime import datetime               # real dates, so 'Mar19' sorts after 'Apr18' instead of alphabetically
import requests                             # makes HTTP calls: fetch a page, download a file, call an API
import pandas as pd                         # the table library; a DataFrame is a spreadsheet you query in code
from bs4 import BeautifulSoup               # parses HTML into a tree you can search for links

DATA = Path("../data")                      # repo-root/data, relative to this notebook in src/
RTT_ZIPS = DATA / "rtt_zips"                # raw monthly zips as downloaded, ~330 MB
RTT_CSV = DATA / "rtt_csv"                  # csvs extracted from those zips, ~8.2 GB
REF = DATA / "reference"                    # trust -> ICB lookup table
KH03 = DATA / "kh03"                        # quarterly bed spreadsheets
for d in (RTT_ZIPS, RTT_CSV, REF, KH03):    # create all four up front
    d.mkdir(parents=True, exist_ok=True)    # parents=True builds ../data itself; exist_ok = no-op on rerun

HEADERS = {"User-Agent": "Mozilla/5.0"}     # nhs.uk returns 403 to the default python-requests agent


## 1. Build the RTT Full CSV manifest

Each financial year (2018-19 ... 2025-26) has one page listing that year's monthly files. We scrape all eight pages, keep only links whose filename says Full-CSV and ends in .zip, and key them by the month token in the filename (e.g. `Mar19`). When a month appears twice (original + revised), the revised link wins.

Cross-checked: this filter picks exactly the official links, e.g. `Full-CSV-data-file-Mar19-revised-ZIP-3506K.zip` and `Full-CSV-data-file-Mar26-ZIP-3M-revised.zip`. Ignore the size hint in the filename — the Mar22 revised file says '111805K' but is actually ~4.5 MB on the server; sizes vary wildly month to month and that's fine.

In [2]:
YEARS = [f"{y}-{str(y + 1)[2:]}" for y in range(2018, 2026)]   # '2018-19' ... '2025-26', the url slugs

manifest = {}                                          # month key like 'Mar19' -> download url
for yr in YEARS:                                       # one page per financial year, 8 total
    page = f"https://www.england.nhs.uk/statistics/statistical-work-areas/rtt-waiting-times/rtt-data-{yr}/"
    soup = BeautifulSoup(requests.get(page, headers=HEADERS).text, "html.parser")   # fetch + parse html
    for a in soup.find_all("a", href=True):            # every <a href=...> link on the page
        href = a["href"]
        name = href.split("/")[-1]                     # filename part of the url
        if "full-csv" not in name.lower() or not name.endswith(".zip"):
            continue                                   # skip the XLS views and commissioner files
        key = re.search(r"([A-Z][a-z]{2}\d{2})", name).group(1)   # grab 'Mar19': 3 letters then 2 digits
        if key not in manifest or "revised" in name.lower():
            manifest[key] = href                       # revised link supersedes the original

months = sorted(manifest, key=lambda k: datetime.strptime(k, "%b%y"))   # %b = 'Mar', %y = '19'
print(len(months), "months:", months[0], "->", months[-1])              # expect 96, Apr18 -> Mar26


96 months: Apr18 -> Mar26


## 2. Download all 96 zips

Streamed in 1 MB chunks so big files don't sit in memory. Each file downloads to a `.part` name and is renamed only once fully written — so a completed filename on disk always means a complete download, and rerunning the cell skips finished files and resumes the rest. File sizes vary a lot month to month; that's expected.

In [3]:
for k in months:                                   # chronological, Apr18 -> Mar26
    url = manifest[k]
    dest = RTT_ZIPS / url.split("/")[-1]           # save under the server's own filename
    if dest.exists():
        continue                                   # already downloaded on a previous run
    tmp = dest.with_suffix(".part")                # write under a temp name while in progress
    r = requests.get(url, headers=HEADERS, stream=True)   # stream=True: body arrives in pieces, not one blob
    with open(tmp, "wb") as f:
        for chunk in r.iter_content(1 << 20):      # 1 << 20 = 2**20 = 1 MB chunks
            f.write(chunk)
    tmp.rename(dest)                               # rename is atomic, so a final filename means a complete file
    print(k, dest.name, f"{dest.stat().st_size / 1e6:.1f} MB")

print("done,", len(list(RTT_ZIPS.glob('*.zip'))), "zips on disk")   # expect 96


Apr18 Full-CSV-data-file-Apr18-revised-ZIP-3380K.zip 3.5 MB


May18 Full-CSV-data-file-May18-revised-ZIP-3424K.zip 3.5 MB


Jun18 Full-CSV-data-file-Jun18-revised-ZIP-3426K.zip 3.5 MB


Jul18 Full-CSV-data-file-Jul18-revised-ZIP-3450K.zip 3.5 MB


Aug18 Full-CSV-data-file-Aug18-revised-ZIP-3464K.zip 3.5 MB


Sep18 Full-CSV-data-file-Sep18-revised-ZIP-K.zip 3.5 MB


... (86 outputs left out)

Dec25 Full-CSV-data-file-Dec25-ZIP-3M-revised.zip 3.5 MB


Jan26 Full-CSV-data-file-Jan26-ZIP-4M-WL5BiP.zip 3.6 MB


Feb26 Full-CSV-data-file-Feb26-ZIP-4M-9j03fJT.zip 3.6 MB


Mar26 Full-CSV-data-file-Mar26-ZIP-3M-revised.zip 3.6 MB
done, 96 zips on disk


## 3. Extract every zip into one csv folder

In [4]:
for z in sorted(RTT_ZIPS.glob("*.zip")):           # each zip holds exactly one month's csv
    zipfile.ZipFile(z).extractall(RTT_CSV)         # dump them all into one flat folder

print(len(list(RTT_CSV.glob('*.csv'))), "csvs extracted")   # expect 96, ~7.7 GB total


96 csvs extracted


## 4. Trust -> ICB reference (ODS ORD API)

Two steps: one call lists every organisation with primary role RO197 (= NHS Trust), then one call per trust fetches its full record, from which we take the active RE5 ('is commissioned by') relationship — its target is the trust's ICB, which carries primary role **RO261** in ODS. Verified live: e.g. North London NHS FT -> `Z9B2Z` (NHS West and North London ICB).

The list returns ~550 trusts of which ~250 are Active; we keep the inactive ones too since they appear in the 2018-era RTT months (their `icb_code` stays empty). ~550 small requests, a couple of minutes total.

In [5]:
# ODS is NHS England's organisation registry; ORD is its REST API. The RTT file gives a provider
# code and nothing else, so sector and commissioner have to come from here.
ORD = "https://directory.spineservices.nhs.uk/ORD/2-0-0"

trusts = requests.get(
    f"{ORD}/organisations",
    params={"PrimaryRoleId": "RO197",       # RO197 = NHS Trust, current and historic
            "Limit": 1000,                  # ~550 exist, so one page covers it
            "_format": "json"},             # otherwise it returns xml
    headers=HEADERS,
).json()["Organisations"]                   # .json() parses the body; ["Organisations"] unwraps the envelope
print(len(trusts), "trusts")                # expect ~544


544 trusts


In [6]:
rows = []                                          # one dict per trust
for t in trusts:                                   # ~544 trusts -> ~544 requests, a couple of minutes
    code = t["OrgId"]                              # provider code, e.g. 'R0A' - same key as in the RTT csvs
    rec = requests.get(f"{ORD}/organisations/{code}", headers=HEADERS).json()["Organisation"]   # full record
    icb_code = ""                                  # default: no ICB, true for long-dissolved trusts
    for rel in rec.get("Rels", {}).get("Rel", []):  # .get twice: some orgs have no Rels block at all
        tgt = rel["Target"]
        # RE5 = 'is commissioned by'; the live one whose target is an ICB (role RO261)
        if rel["Status"] == "Active" and rel["id"] == "RE5" and tgt["PrimaryRoleId"]["id"] == "RO261":
            icb_code = tgt["OrgId"]["extension"]   # the ICB's own ods code, e.g. 'QOP'
    rows.append({
        "trust_code": code,                        # join key to RTT 'Provider Org Code'
        "trust_name": t["Name"],
        "status": t["Status"],                     # Active / Inactive
        "icb_code": icb_code,                      # empty for dissolved trusts - expected, not an error
        "postcode": rec["GeoLoc"]["Location"].get("PostCode", ""),   # for the geography tier
    })

ref = pd.DataFrame(rows)                           # ~544 rows x 5 cols
ref.to_csv(REF / "trust_icb.csv", index=False)
ref.head()


,trust_code,trust_name,status,icb_code,postcode
0,G6V2S,NORTH LONDON NHS FOUNDATION TRUST,Active,Z9B2Z,NW1 0PE
1,K0N6A,THE ONLINE NHS TRUST,Active,QKK,SE1 8UG
2,R0A,MANCHESTER UNIVERSITY NHS FOUNDATION TRUST,Active,QOP,M13 9WL
3,R0B,SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION T...,Active,QHM,SR4 7TP
4,R0C,PROJECT NIGHTINGALE NHS TRUST,Inactive,,E16 1XL


## 5. KH03 overnight beds 

The bed-availability page links every quarterly overnight spreadsheet back to 2010 — 61 files as of Jul 2026, latest `Beds-Open-Overnight-Web_File-Q4-2025-26.xlsx` (verified live). We take any xls/xlsx link with 'night' in the filename and download whatever isn't already on disk.

In [7]:
page = "https://www.england.nhs.uk/statistics/statistical-work-areas/bed-availability-and-occupancy/bed-data-overnight/"
soup = BeautifulSoup(requests.get(page, headers=HEADERS).text, "html.parser")

links = sorted({                                                        # a set, so repeated links collapse
    a["href"] for a in soup.find_all("a", href=True)
    if a["href"].split("/")[-1].lower().endswith((".xls", ".xlsx"))     # spreadsheets only
    and "night" in a["href"].split("/")[-1].lower()                     # 'overnight', not day-case beds
})
print(len(links), "files")                          # expect 61, back to 2011

for url in links:
    dest = KH03 / url.split("/")[-1]                # keep the server's filename
    if dest.exists():
        continue
    dest.write_bytes(requests.get(url, headers=HEADERS).content)   # a few hundred KB each
    print(dest.name)


61 files


Beds-Open-Overnight-Web_File-Final-Q1-2015-16-52341.xlsx


Beds-Open-Overnight-Web_File-Final-Q2-2015-16-38498.xlsx


Beds-Open-Overnight-Web_File-Final-Q3-2015-16-39849.xlsx


Beds-Open-Overnight-Web_File-Q1-2011-12-November-2012-Refresh-v3-Final.xls


Beds-Open-Overnight-Web_File-Q1-2012-13-May-2013-Refresh-Final-47384.xls


... (52 outputs left out)

Beds-Open-Overnight-Web_File-Q2-2025-26.xlsx


Beds-Open-Overnight-Web_File-Q3-2024-25-revised.xlsx


Beds-Open-Overnight-Web_File-Q3-2025-26.xlsx


Beds-Open-Overnight-Web_File-Q4-2025-26.xlsx
